# TP-1 — TinyGPT: pretraining, decodificación y Mixture of Experts

Autor: Msc. Abraham R.

Vas a preentrenar un GPT muy pequeño (un decoder transformer) a nivel de caracteres sobre
Shakespeare, y después trabajarlo en dos direcciones.

## Arquitecturas de TinyGPT

Pensado para la [materia NLP-II](https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/CEIA-LLMIAG),
donde tratamos arquitecturas y teoría, vamos a empezar con un modelo denso, equivalente y
basado en GPT-2, para terminar en un modelo que consiste en un **GPT de Mixture of
Experts**, equivalente a modelos como:
- DeepSeek-MoE/v2/v3/R1
- Qwen-MoE
- Mistral/Mixtral

**Parte 1 — Inferencia.** El modelo te da una distribución de probabilidad sobre el
siguiente carácter. *Elegir* un carácter a partir de ella es una decisión de diseño
aparte, y cambia la salida mucho más de lo que lo haría otra época de entrenamiento.

**Parte 2 — Arquitectura.** Reemplazá el bloque feed-forward denso por un Mixture of
Experts, y medí qué te da y qué te cuesta.

Se entrena un solo modelo, una sola vez, al principio, y se reutiliza en todo el trabajo.

## Consignas

**Parte 1**
- Consigna I — implementar `generateV2`: greedy, temperatura, top-k y top-p.
- Consigna II — comparar las estrategias de decodificación y explicar qué hace cada perilla.
- Consigna III — medir qué te da realmente el KV-cache. No asumas que gana.
- Consigna IV — leer e interpretar los mapas de atención de tu modelo entrenado.

**Parte 2**
- Consigna V — implementar ruteo top-k en `MoELayer.forward`.
- Consigna VI — entrenar TinyGPT-MoE y compararlo contra el baseline denso.
- Consigna VII — medir la utilización de expertos y diagnosticar colapso de ruteo.
- Consigna VIII — implementar [DeepSeekMoE](https://arxiv.org/pdf/2401.06066) y medirlo.
- Consigna IX — tu MoE es mucho más lento por paso que el modelo denso. Hacelo más rápido.

**Después**
- Consigna X — preentrenar el modelo *denso* con un tokenizer subword real de GPT-2 y
  comparar como corresponde. El TP-2 y el notebook del TP-3 dependen del checkpoint que
  produce esta consigna.
- *Opcional* — una loss auxiliar de balanceo de carga.

## Cómo trabajar en esto

Toda consigna de la que dependa una celda posterior viene con una celda `check_*()`.
Corréla antes de seguir — cuesta segundos y atrapa los bugs que si no aparecerían como
una curva de loss plana media hora más tarde. Que pasen todos los self-checks es el mínimo
para entregar.



In [ ]:
import time
from typing import List, Optional

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader

from tinygpt import (
    CharDataset,
    CharTokenizer,
    GPTConfig,
    MoEArgs,
    count_parameters,
    describe,
    free,
    load_checkpoint,
    get_device,
    load_tinyshakespeare,
    plot_losses,
    resume,
    run_training,
    trim_kv_cache,
    visualize_attention,
)
from trainer import Trainer

torch.manual_seed(1337)

## Retomar después de reiniciar el kernel

Este notebook entrena cuatro modelos y los mantiene en memoria, así que un reinicio los
pierde a todos y toda celda de acá para abajo falla con `NameError`. Los pesos y las curvas
de loss están en disco, así que no hace falta reentrenar — corré la celda de abajo y saltá
hasta donde estabas.

En una corrida desde cero no la toques: sólo restaura lo que ya está terminado.


In [ ]:
def resume_all():
    """
    Reconstruye los modelos que ya tengan un checkpoint terminado.

    Corré primero las celdas de arriba -- las definiciones son instantáneas, lo único que no
    lo es es el entrenamiento -- y después llamá a esto. Restaura cada modelo y su curva de
    loss, y saltea todo aquello cuya clase o config todavía no esté en memoria.
    """
    import os

    g = globals()
    specs = [
        ("model",     "./checkpoints/tp1_dense",    "history",     ("config",)),
        ("moe_model", "./checkpoints/tp1_moe",      "history_moe", ("moe_config",)),
        ("ds_model",  "./checkpoints/tp1_deepseek", "history_ds",  ("ds_config",)),
        ("sub_model", "./checkpoints/tp1_subword",  "sub_history", ("sub_config",)),
    ]

    made, skipped = [], []
    for name, path, hist, needs in specs:
        if not os.path.exists(os.path.join(path, "checkpoint_final.pt")):
            continue
        if any(n not in g for n in needs):
            skipped.append(f"{name} (necesita {', '.join(needs)})")
            continue
        # reutiliza el objeto si la celda de config ya construyó uno sin entrenar
        m = g.get(name) or g["TinyGPT"](g[needs[0]]).to(g["device"])
        g[name], g[hist] = m, g["resume"](m, path, map_location=g["device"])
        made.append(name)

    print("restaurados:", ", ".join(made) if made else "nada")
    if skipped:
        print("todavía no:", "; ".join(skipped))


# resume_all()   # descomentar después de reiniciar el kernel


## Descarga del dataset

In [ ]:
# Bajá esto si el entrenamiento es lento en tu máquina.
N_CHARS = 100_000

text = load_tinyshakespeare(N_CHARS)
print(text[:400])
print(f"...\n\n{len(text):,} caracteres")

## Codificación a nivel de caracteres

`CharTokenizer` mapea cada carácter distinto a un id entero. Un carácter == un token.

In [ ]:
tokenizer = CharTokenizer(text)
print(f"vocab_size = {tokenizer.vocab_size}")
print(repr("".join(tokenizer.chars)))

data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

# Split de entrenamiento/validación
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"entrenamiento {train_data.shape} | validación {val_data.shape}")

## Configuración del GPT

`ff_class` le permite a la Parte 2 cambiar el bloque feed-forward por un MoE sin tocar el
código del modelo.

In [ ]:
config = GPTConfig(vocab_size=tokenizer.vocab_size)
print(config)

## Dataloaders

In [ ]:
# Usá 0 en macOS/MPS; unos pocos workers ayudan en CUDA.
NUM_WORKERS = 0

train_dataset = CharDataset(train_data, config.block_size)
val_dataset = CharDataset(val_data, config.block_size)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                          drop_last=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False,
                        drop_last=True, pin_memory=True, num_workers=NUM_WORKERS)

print(f"{len(train_loader)} batches de entrenamiento | {len(val_loader)} de validación")

# La arquitectura

Las próximas tres celdas son todo el modelo. *Leelas antes de correrlas*. El resto de la
materia se apoya en este código, y el TP-2 lo importa desde `tinygpt.py` en vez de
redefinirlo.

Hay tres detalles que merecen tu atención, porque son los más fáciles de tener mal en
silencio:

1. Todo `forward` devuelve la misma 3-tupla `(out, kv_cache, weights)`. Decidir la forma
   del retorno según *si te pasaron un cache o no* es un bug clásico: el primer paso de
   decodificación no tiene cache para pasar, así que el cache nunca se activa y
   `use_cache=True` queda en un no-op del que nunca te enterás.
2. La máscara causal y los positional embeddings están ambos desplazados por el largo del
   cache. Con cache el modelo ve un token nuevo, pero ese token no está en la posición 0.
3. Todas las cabezas se proyectan y atienden en un único matmul batcheado, no en un loop
   de Python sobre módulos por cabeza. A esta escala el tiempo de reloj está dominado por
   los lanzamientos de kernel más que por la aritmética, así que un loop sobre `n_head`
   módulos cuesta varias veces el resto del paso — que es el mismo efecto que vas a medir,
   y arreglar, en la Consigna IX.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Self-atención causal multi-cabeza, batcheada a lo largo de las cabezas.

    Todas las cabezas se proyectan en un solo matmul y atienden en un solo matmul
    batcheado, en vez de iterar sobre un módulo `AttentionHead` por cabeza. La versión con
    loop es más fácil de leer pero lanza `n_head` kernels chicos por capa, y a este tamaño
    de modelo el tiempo de reloj está dominado por los lanzamientos de kernel más que por
    la aritmética -- así que el loop costaba varias veces todo el resto del paso.

    La atención sigue escrita a mano (scores, máscara causal, softmax) en vez de llamar a
    F.scaled_dot_product_attention, porque la máscara es justamente lo que se está
    enseñando y SDPA no puede devolver los pesos de atención para la consigna de
    visualización.

    Todo forward devuelve la misma 3-tupla `(out, kv_cache, weights)`; las entradas
    finales son `None` cuando no se piden.
    """

    def __init__(self, args: GPTConfig):
        super().__init__()
        assert args.n_embd % args.n_head == 0, "n_embd tiene que ser divisible por n_head"
        self.n_head = args.n_head
        self.head_dim = args.n_embd // args.n_head

        # Una sola proyección para todas las cabezas: (B, T, C) -> (B, T, 3C), partida en K, Q, V.
        self.key_query_value = nn.Linear(args.n_embd, 3 * args.n_embd, bias=args.bias)
        self.proj = nn.Linear(args.n_embd, args.n_embd, bias=args.bias)

        self.attn_dropout = nn.Dropout(args.dropout)
        self.dropout = nn.Dropout(args.dropout)
        self.register_buffer(
            "tril", torch.tril(torch.ones(args.block_size, args.block_size))
        )

    def _split(self, t, B, T):
        """(B, T, C) -> (B, n_head, T, head_dim)"""
        return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

    def forward(self, x, kv_cache=None, use_cache=False, return_weights=False):
        B, T, C = x.shape
        k, q, v = self.key_query_value(x).chunk(3, dim=-1)
        k, q, v = self._split(k, B, T), self._split(q, B, T), self._split(v, B, T)

        if kv_cache is not None:
            k_prev, v_prev = kv_cache.unbind(dim=0)      # (B, n_head, T_past, head_dim)
            k = torch.cat((k_prev, k), dim=2)
            v = torch.cat((v_prev, v), dim=2)

        new_cache = torch.stack((k, v)) if use_cache else None

        T_k = k.size(2)          # total de keys atendidas
        past = T_k - T           # cuántas de ellas vinieron del cache

        wei = q @ k.transpose(-2, -1) * (self.head_dim ** -0.5)   # (B, n_head, T, T_k)
        # Las filas están desplazadas por `past` para que la máscara siga siendo correcta
        # con KV-cache: la query en la posición absoluta `past + t` puede ver las keys
        # 0..past + t.
        wei = wei.masked_fill(self.tril[past:T_k, :T_k] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)

        out = (wei @ v).transpose(1, 2).contiguous().view(B, T, C)
        out = self.dropout(self.proj(out))

        # (n_head, B, T, T_k) -- el layout que espera visualize_attention
        weights = wei.transpose(0, 1) if return_weights else None
        return out, new_cache, weights

In [ ]:
class FeedForward(nn.Module):
    """MLP densa: el módulo que la Parte 2 reemplaza por un MoE."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.ReLU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Bloque decoder transformer pre-norm."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = MultiHeadAttention(config)

        ff_class = config.ff_class if config.ff_class is not None else FeedForward
        self.ff = ff_class(config)

    def forward(self, x, kv_cache=None, use_cache=False, return_weights=False):
        attn_out, new_cache, weights = self.attn(
            self.ln1(x), kv_cache=kv_cache, use_cache=use_cache, return_weights=return_weights
        )
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x, new_cache, weights

## TinyGPT

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        if config.tie_weights:
            self.head.weight = self.token_emb.weight

    def forward(self, idx, kv_cache=None, use_cache=False, return_weights=False):
        """
        Args:
            idx:            ids de tokens (B, T).
            kv_cache:       lista de caches por capa de una llamada anterior, o None.
            use_cache:      devolver un KV-cache actualizado junto con los logits.
            return_weights: devolver los mapas de atención por capa junto con los logits.

        Returns:
            `logits` cuando ambos flags son False (esto es lo que espera `Trainer`),
            si no, una tupla `(logits[, kv_cache][, attn_weights])`.

        Notá que los flags determinan la forma del retorno -- NO el hecho de que
        `kv_cache` sea None o no. Derivarla de la entrada es la manera en que un
        cache nunca se activa en silencio: el primer paso de decodificación no
        tiene cache para pasar.
        """
        B, T = idx.shape
        past_len = kv_cache[0].shape[-2] if kv_cache is not None else 0
        assert past_len + T <= self.config.block_size, (
            f"la secuencia de {past_len + T} excede block_size={self.config.block_size}"
        )

        tok_emb = self.token_emb(idx)
        # Las posiciones tienen que continuar desde donde quedó el cache; si no, cada
        # paso cacheado reutiliza la posición 0.
        pos = torch.arange(past_len, past_len + T, device=idx.device)
        x = tok_emb + self.pos_emb(pos)[None, :, :]

        new_kv_cache = [] if use_cache else None
        all_weights = [] if return_weights else None

        for i, block in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, updated_kv, weights = block(
                x, kv_cache=layer_cache, use_cache=use_cache, return_weights=return_weights
            )
            if use_cache:
                new_kv_cache.append(updated_kv)
            if return_weights:
                all_weights.append(weights)

        logits = self.head(self.ln_f(x))

        if not use_cache and not return_weights:
            return logits
        out = [logits]
        if use_cache:
            out.append(new_kv_cache)
        if return_weights:
            out.append(all_weights)
        return tuple(out)

    @torch.no_grad()
    def resize_token_embeddings(self, new_vocab_size: int) -> None:
        """
        Agranda el embedding de entrada y la cabeza de salida hasta `new_vocab_size`,
        preservando todas las filas preentrenadas.

        Hace falta cuando el fine-tuning agrega tokens especiales (marcadores de
        chat/instrucciones) que el vocabulario de preentrenamiento no tenía.
        """
        old_vocab, n_embd = self.token_emb.weight.shape
        if new_vocab_size == old_vocab:
            return
        assert new_vocab_size > old_vocab, "achicar el vocabulario no está soportado"

        device = self.token_emb.weight.device
        dtype = self.token_emb.weight.dtype

        new_emb = nn.Embedding(new_vocab_size, n_embd, device=device, dtype=dtype)
        new_emb.weight.normal_(mean=0.0, std=0.02)
        new_emb.weight[:old_vocab] = self.token_emb.weight
        self.token_emb = new_emb

        new_head = nn.Linear(n_embd, new_vocab_size, bias=False, device=device, dtype=dtype)
        new_head.weight.normal_(mean=0.0, std=0.02)
        new_head.weight[:old_vocab] = self.head.weight
        self.head = new_head

        self.config.vocab_size = new_vocab_size

## Generación baseline (inferencia)

Esta es la función que vas a mejorar en la Consigna I. Muestrea del softmax *completo* a
temperatura 1.0 — sin truncamiento, sin control de temperatura.

In [ ]:
@torch.no_grad()
def generate(
    model: nn.Module,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 100,
    use_cache: bool = True,
    device: Optional[str] = None,
) -> str:
    """
    Decodificación baseline: muestrear del softmax completo a temperatura 1.0.

    Esta es la función que el TP-1 te pide mejorar con greedy / temperatura /
    top-k / top-p.
    """
    model.eval()
    device = device or next(model.parameters()).device
    block_size = model.config.block_size

    idx = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device)[None, :]
    kv_cache = None

    for _ in range(max_new_tokens):
        if kv_cache is None:
            idx_cond = idx[:, -block_size:]
        else:
            idx_cond = idx[:, -1:]

        out = model(idx_cond, kv_cache=kv_cache, use_cache=use_cache)
        if use_cache:
            logits, kv_cache = out
            # Nunca dejes que el cache crezca más que la ventana de contexto.
            kv_cache = trim_kv_cache(kv_cache, block_size - 1)
        else:
            logits = out

        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_token), dim=1)

    return tokenizer.decode(idx[0].tolist())

# Setup

In [ ]:
device = get_device()
print(f"dispositivo: {device}")

model = TinyGPT(config).to(device)
describe(model, "TinyGPT (denso)")

# torch.compile tracea el modelo y fusiona kernels. Es una mejora real en CUDA, es poco
# confiable en MPS (crashea), y da poco a este tamaño de modelo -- así que se habilita
# sólo donde ayuda.
#
# Ojo con lo que le hace a los checkpoints: compilar envuelve al módulo, así que toda
# clave del state-dict gana un prefijo "_orig_mod.". Por eso el TP-2 y el notebook de
# serving cargan los pesos vía tinygpt.load_checkpoint, que lo saca.
if device == "cuda":
    model = torch.compile(model)
    print("torch.compile habilitado")


In [ ]:
EPOCHS = 2


def make_optimizer(params, lr: float = 1e-3):
    """
    Adam de 8 bits en CUDA, el AdamW de torch en todo lo demás.

    bitsandbytes mantiene el estado del optimizador en 8 bits en vez de 32, lo que reduce
    aproximadamente a la mitad la memoria que necesita Adam -- irrelevante con 800k
    parámetros, y la razón por la que podés hacer fine-tuning de un modelo de 7B en una
    sola GPU.

    Todas las corridas de este notebook lo usan, así que la comparación denso/MoE nunca
    cambia dos cosas a la vez.
    """
    if device == "cuda":
        try:
            from bitsandbytes.optim import AdamW as bnbAdamW
            return bnbAdamW(params, lr=lr)
        except ImportError:
            print("bitsandbytes no está instalado (pip install bitsandbytes) -- se usa el AdamW de torch")
    return AdamW(params, lr=lr)


optimizer = make_optimizer(model.parameters())
scheduler = StepLR(optimizer, step_size=100, gamma=0.9)
loss_fn = torch.nn.CrossEntropyLoss()


# Entrenamiento

Aproximadamente un minuto por época en GPU, más en CPU. Bajá `N_CHARS` arriba si necesitás
que sea más rápido.

## Qué corre distinto en CUDA

Hay tres optimizaciones que se habilitan automáticamente cuando hay un dispositivo CUDA
presente, y se saltean si no. Ninguna cambia la matemática — cambian lo que el hardware
hace con ella.

| | qué hace | por qué es sólo para CUDA |
|---|---|---|
| **Adam de 8 bits** (`make_optimizer`) | estado del optimizador en 8 bits en vez de 32, lo que reduce a la mitad la memoria de Adam | `bitsandbytes` no tiene build para Apple Silicon |
| **`torch.compile`** | tracea el modelo y fusiona kernels | poco confiable en MPS; crashea |
| **autocast en bfloat16** (`run_training`) | la mitad de memoria de activaciones, matmuls más rápidos en tensor cores | el autocast de MPS está incompleto y `GradScaler` no soporta ese backend |

A este tamaño de modelo ninguna importa demasiado. Importan enormemente a la escala en la
que realmente las usarías, y por eso están acá en vez de omitidas: Adam de 8 bits es la
diferencia entre poder hacer fine-tuning de un modelo de 7B en una sola GPU y no poder.

`run_training(..., use_amp=True)` fuerza precisión mixta si querés ver qué hace en tu
hardware.

## Una nota sobre memoria

Este notebook entrena cuatro modelos en un mismo kernel y los mantiene vivos, porque las
celdas posteriores los comparan. Ni CUDA ni MPS devuelven al sistema operativo los bloques
liberados por su cuenta, y los gathers de forma variable del MoE fragmentan feo el
allocator — así que sin ayuda el kernel crece hasta que la máquina empieza a hacer swap.

`free()` libera el cache; `run_training` la llama después de cada corrida, y las celdas de
abajo la vuelven a llamar después de soltar un trainer. Si aun así te quedás sin memoria,
reiniciá el kernel y bajá `N_CHARS`.


In [ ]:
trainer = Trainer(
    model=model,
    train_data_loader=train_loader,
    test_data_loader=val_loader,
    loss_fn=loss_fn,
    gradient_accumulation_steps=1,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir="./checkpoints/tp1_dense",
    save_every_n=500,
)

history = run_training(trainer, epochs=EPOCHS)
plot_losses({"denso": history}, title="Preentrenamiento de TinyGPT")

# El trainer retiene el estado del optimizador; el modelo en sí todavía hace falta.
del trainer
free()

### Prueba rápida

In [ ]:
print(generate(model, tokenizer, "To be", max_new_tokens=300, use_cache=True))

# Consigna I — estrategias de decodificación

El modelo te da una distribución de probabilidad sobre el siguiente carácter. *Elegir* un
carácter de esa distribución es una decisión de diseño aparte, y cambia la salida mucho más
de lo que lo haría otra época de entrenamiento.

Implementá `generateV2` abajo, soportando:

- Decodificación greedy (elegir el token de máxima probabilidad).
- Muestreo con temperatura.
- Muestreo top-k o top-p.

Aplicá primero la temperatura, después top-k, después top-p, y recién ahí renormalizá y
muestreá. `top_k=None` y `top_p=None` significan sin truncamiento, y `do_sample=False`
ignora todas las demás perillas.

### Referencias
- [huggingface generate](https://huggingface.co/docs/transformers/main_classes/text_generation)
- [The Curious Case of Neural Text Degeneration (top-p)](https://arxiv.org/abs/1904.09751)


In [ ]:
# TODO: implementar decodificación greedy, temperatura, top-k y top-p.
@torch.no_grad()
def generateV2(
    model: nn.Module,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 300,
    use_cache: bool = True,
    do_sample: bool = True,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    device: Optional[str] = None,
) -> str:
    """
    Args:
        do_sample:   False -> greedy (arg-max); se ignoran todas las demás perillas.
        temperature: >0. Más bajo es más determinista, más alto es más aleatorio.
        top_k:       conservar los k tokens más probables, o None para no truncar.
        top_p:       conservar el conjunto más chico cuya probabilidad acumulada sea >= p,
                     o None para no truncar.

    Returns:
        El prompt más `max_new_tokens` caracteres generados.
    """
    raise NotImplementedError("Consigna I")

## Self-check

Tres propiedades que tienen que cumplirse si tu implementación está bien. Corré esto antes
de seguir — atrapa la mayoría de los bugs.

In [ ]:
def self_check():
    kw = dict(model=model, tokenizer=tokenizer, prompt="To be")

    # 1. greedy es determinista
    a = generateV2(**kw, max_new_tokens=60, do_sample=False)
    b = generateV2(**kw, max_new_tokens=60, do_sample=False)
    assert a == b, "la decodificación greedy no es determinista"

    # 2. top_k=1 colapsa la distribución sobre el arg-max, así que tiene que dar igual que greedy
    torch.manual_seed(0)
    c = generateV2(**kw, max_new_tokens=60, do_sample=True, top_k=1)
    assert c == a, "top_k=1 debería reducirse a greedy"

    # Los dos siguientes miran un solo paso: a lo largo de muchos pasos, un casi-empate
    # entre los dos logits más altos termina rompiendo la igualdad exacta, lo que no
    # prueba nada.
    one = generateV2(**kw, max_new_tokens=1, do_sample=False)

    # 3. temperatura -> 0 vuelve al softmax one-hot en el arg-max
    torch.manual_seed(0)
    assert generateV2(**kw, max_new_tokens=1, do_sample=True, temperature=1e-3) == one, \
        "temperatura -> 0 debería acercarse a greedy"

    # 4. top_p -> 0 tiene que dejar exactamente un token: el más probable. Este es el
    #    off-by-one de las pistas -- una comparación mal puesta deja el conjunto vacío acá.
    torch.manual_seed(0)
    assert generateV2(**kw, max_new_tokens=1, do_sample=True, top_p=1e-9) == one, \
        "top_p -> 0 debería dejar únicamente el token más probable"

    print("todos los checks pasaron")


self_check()

# Consigna II — comparar las estrategias

Corré la grilla de abajo, leé las muestras, y contestá las preguntas que siguen.

In [ ]:
PROMPT = "To be"

SETTINGS = [
    ("greedy",              dict(do_sample=False)),
    ("muestreo puro (T=1)", dict(do_sample=True)),
    ("T=0.5",               dict(do_sample=True, temperature=0.5)),
    ("T=1.5",               dict(do_sample=True, temperature=1.5)),
    ("top_k=10",            dict(do_sample=True, top_k=10)),
    ("top_p=0.9",           dict(do_sample=True, top_p=0.9)),
    ("T=0.8 + top_p=0.9",   dict(do_sample=True, temperature=0.8, top_p=0.9)),
]

for name, kwargs in SETTINGS:
    torch.manual_seed(0)
    out = generateV2(model, tokenizer, PROMPT, max_new_tokens=200, **kwargs)
    print("=" * 70)
    print(f"### {name}")
    print(out)

## Preguntas

Respondé en esta celda.

1. La decodificación greedy degenera de una manera muy específica. Describí qué pasa y
   explicá *por qué* la política de arg-max lo produce.
2. `T=1.5` y `T=0.5` fallan en direcciones opuestas. Nombrá los dos modos de falla.
3. Top-k y top-p truncan ambos la distribución. Dá una situación concreta donde un `k`
   fijo sea la elección equivocada y top-p se adapte mejor. (Pensá en qué tan picuda es la
   distribución después de `Th` versus después de un espacio.)
4. ¿Qué configuración pondrías en producción, y por qué?

---

-
-
-
-

# Consigna III — ¿qué te da realmente el KV-cache?

### Teoría
Sin cache, generar el token *n* vuelve a correr la atención sobre los *n* tokens
anteriores. Con cache, las keys y values de esos tokens se reutilizan y sólo se proyecta el
token nuevo. Asintóticamente eso convierte una generación O(N²) en O(N).

### Consigna
**Medí si la teoría se cumple realmente acá, y no asumas que el cache gana** — reportá el
número que te dé, sea el que sea, y después explicalo.

In [ ]:
# TODO: cronometrar generateV2 con y sin KV-cache, para varios largos.
def benchmark_cache(lengths=(50, 100, 200, 400), repeats: int = 3):
    """
    Para cada largo, cronometrar la generación con use_cache=True y use_cache=False e
    imprimir el speed-up. Graficar tokens vs. tiempo de reloj para ambos.
    """
    raise NotImplementedError("Consigna III")


benchmark_cache()

## Preguntas

1. ¿El cache hizo la generación más rápida? Reportá tus números. Si no la hizo más rápida,
   explicá adónde se va realmente el tiempo: contá los lanzamientos de kernel y las
   asignaciones de tensores que hace esta implementación por paso de decodificación, y
   pesalos contra la aritmética que el cache ahorra.
2. Asintóticamente el cache es una ganancia clara. Nombrá las propiedades específicas de
   *esta* implementación y de *este* tamaño de modelo que esconden esa ganancia, y describí
   un escenario (tamaño de modelo, largo de secuencia, implementación de atención) donde se
   vería con claridad.
3. ¿Qué te cuesta el cache en memoria? Dá la fórmula en términos de `n_layer`, `n_head`,
   `head_dim`, tamaño de batch y largo de secuencia, y evaluala para este modelo.
4. La decodificación con y sin cache produce texto idéntico mientras la secuencia entra en
   la ventana de contexto, y diverge después. ¿Por qué?

---

-
-
-
-

# Consigna IV — visualizar la atención

Un GPT completa texto. Veamos a qué atienden realmente las cabezas de tu modelo
entrenado.

In [ ]:
visualize_attention(model, tokenizer, "To be or not to be")

## Preguntas

1. Todos los mapas son triangulares inferiores. ¿Qué línea de `AttentionHead.forward` causa
   eso, y qué se rompería si la sacaras?
2. Encontrá una cabeza que atienda mayormente al carácter inmediatamente anterior y otra
   que reparta su masa más ampliamente. ¿Qué está haciendo cada una, plausiblemente?

---

-
-
-

---

# Parte 2 — Arquitectura

La Parte 1 cambió cómo *muestreás* de un modelo entrenado. La Parte 2 cambia el modelo
mismo.

Vas a reemplazar el bloque feed-forward denso por un Mixture of Experts — la idea detrás de
Mixtral, DeepSeek y Qwen-MoE — y compararlo contra el modelo que acabás de entrenar. No hay
que reentrenar el baseline: el `model` y la `history` de la Parte 1 son la comparación.

En un transformer denso todo token pasa por la *misma* red feed-forward. Un MoE reemplaza
esa única FFN por `E` expertos y una compuerta chica (*gate*) que elige `k` de ellos por
token. Los parámetros totales crecen con `E`; los parámetros *usados* para un token dado
crecen sólo con `k`. Comprás capacidad sin pagar el cómputo correspondiente.

Los costos también son reales, y los vas a medir: la gate puede colapsar sobre unos pocos
expertos y dejar al resto sin nada, el ruteo es discreto así que los gradientes sólo llegan
a los expertos que fueron seleccionados, y una implementación ingenua es *más lenta* que la
densa a esta escala.

### Referencias
- [Mixtral of Experts](https://arxiv.org/abs/2401.04088)
- [Outrageously Large Neural Networks (sparsely-gated MoE)](https://arxiv.org/abs/1701.06538)
- [Switch Transformers](https://arxiv.org/abs/2101.03961)
- `Clase_IV_MoEs.pdf`


In [ ]:
def build_trainer(model, save_dir, lr=1e-3):
    """Mismo optimizador, scheduler y loss en toda corrida, así lo único que cambia es la arquitectura."""
    optimizer = make_optimizer(model.parameters(), lr=lr)
    scheduler = StepLR(optimizer, step_size=100, gamma=0.9)
    return Trainer(
        model=model,
        train_data_loader=train_loader,
        test_data_loader=val_loader,
        loss_fn=torch.nn.CrossEntropyLoss(),
        gradient_accumulation_steps=1,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        save_dir=save_dir,
        save_every_n=500,
    )

## Los bloques de construcción

Dos piezas, ambas dadas.

Un experto es una FFN: la misma pila `Linear -> ReLU -> Linear -> Dropout` que usa el
modelo denso. Nada exótico — lo interesante es a quién se rutea hacia él.

La gate puntúa cada experto para cada token, mapeando `(B, T, n_embd)` a
`(B, T, num_experts)`. Un solo `nn.Linear`, sin bias, sin activación; el softmax pasa
después, y sólo sobre los scores del top-k.

`Expert` toma su ancho oculto de `config.moe_args.expert_hidden_mult` en vez de hardcodear
4x. Vas a necesitar eso en la Consigna VIII.


In [ ]:
class Expert(nn.Module):
    """
    Una sola MLP experta dentro de una capa MoE.

    El ancho oculto viene de moe_args.expert_hidden_mult (4.0 es lo estándar), para que la
    segmentación fina pueda achicar cada experto sin tocar esta clase.
    """

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        hidden = int(config.moe_args.expert_hidden_mult * config.n_embd)
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, hidden),
            nn.ReLU(),
            nn.Linear(hidden, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Gate(nn.Module):
    """
    Router: puntúa cada experto para cada token. Devuelve logits crudos, no probabilidades.
    """

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.proj = nn.Linear(config.n_embd, config.moe_args.num_experts, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


# Consigna V — ruteo top-k

Este es el núcleo del trabajo. `MoELayer.forward` recibe `x` de forma `(B, T, n_embd)` y
devuelve la misma forma.

La gate puntúa cada experto para cada token; te quedás con los `k` mejores, hacés softmax
sólo sobre esos, corrés cada experto seleccionado, y combinás su salida ponderada por la
gate. El ruteo es por token, no por secuencia — así que aplaná `(B, T, C)` a `(N, C)` antes
de rutear.

Dos cosas para tener bien, porque las dos fallan en silencio:

- El softmax va *después* del top-k, sólo sobre los expertos seleccionados. Normalizar
  antes de truncar deja pesos por token que ya no suman 1.
- Todo tiene que seguir siendo diferenciable, o la gate nunca aprende.

Antes de retornar, guardá la decisión de ruteo en el módulo —
`self.last_indices = indices.detach()` y
`self.last_gate_probs = F.softmax(logits, dim=-1).detach()` — la Consigna VII los lee.

In [ ]:
class MoELayer(nn.Module):
    """
    Capa feed-forward Mixture of Experts con compuerta sparse.
    """

    def __init__(self, experts: List[nn.Module], gate: nn.Module, moe_args: MoEArgs):
        super().__init__()
        self.experts = nn.ModuleList(experts)
        self.gate = gate
        self.args = moe_args
        self.last_indices = None
        self.last_gate_probs = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Consigna V
        raise NotImplementedError


class MoEFFN(nn.Module):
    """
    Reemplazo directo de FeedForward. Se asigna a config.ff_class para que Block lo tome
    sin ningún cambio en el código del modelo.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.moe_args is not None, "hay que setear config.moe_args para usar MoEFFN"
        self.moe = MoELayer(
            experts=[Expert(config) for _ in range(config.moe_args.num_experts)],
            gate=Gate(config),
            moe_args=config.moe_args,
        )

    def forward(self, x):
        return self.moe(x)

## Self-check

Corré esto antes de entrenar. Cuesta segundos y atrapa los bugs de ruteo que si no
aparecerían como una curva de loss plana media hora más tarde.

In [ ]:
def check_moe():
    cfg = GPTConfig(vocab_size=tokenizer.vocab_size, n_embd=32, dropout=0.0,
                    moe_args=MoEArgs(num_experts=4, num_experts_per_token=2))
    layer = MoELayer([Expert(cfg) for _ in range(4)], Gate(cfg), cfg.moe_args)
    layer.eval()
    x = torch.randn(2, 5, 32)

    # 1. se preserva la forma
    out = layer(x)
    assert out.shape == x.shape, f"{out.shape} != {x.shape}"

    # 2. los gradientes llegan tanto a los expertos como a la gate
    out.sum().backward()
    assert layer.gate.proj.weight.grad is not None, "la gate no recibe gradiente"
    assert any(p.grad is not None and p.grad.abs().sum() > 0
               for p in layer.experts.parameters()), "ningún experto recibe gradiente"

    # 3. con k=1 la salida tiene que ser igual a la del experto seleccionado, sin tocar
    cfg1 = GPTConfig(vocab_size=tokenizer.vocab_size, n_embd=32, dropout=0.0,
                     moe_args=MoEArgs(num_experts=4, num_experts_per_token=1))
    top1 = MoELayer([Expert(cfg1) for _ in range(4)], Gate(cfg1), cfg1.moe_args)
    top1.eval()
    with torch.no_grad():
        y = top1(x)
        choice = top1.gate(x).argmax(dim=-1)          # (B, T)
        expected = torch.stack([
            torch.stack([top1.experts[int(choice[b, t])](x[b, t]) for t in range(x.shape[1])])
            for b in range(x.shape[0])
        ])
    err = (y - expected).abs().max().item()
    assert err < 1e-5, f"discrepancia en el ruteo top-1: {err}"

    # 4. el ruteo tiene que ser por token, no por secuencia
    assert top1.last_indices is not None, "guardá last_indices (ver Consigna V)"
    assert top1.last_indices.shape[0] == x.shape[0] * x.shape[1]

    print("todos los checks pasaron")


check_moe()

# Consigna VI — entrenar TinyGPT-MoE

`Block` lee `config.ff_class`, así que convertir TinyGPT en un MoE es un cambio de
configuración, no un cambio de código.

In [ ]:
moe_config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    ff_class=MoEFFN,
    moe_args=MoEArgs(num_experts=4, num_experts_per_token=2),
)

moe_model = TinyGPT(moe_config).to(device)
if device == "cuda":
    moe_model = torch.compile(moe_model)

describe(model, "TinyGPT (denso)")
describe(moe_model, f"TinyGPT-MoE (E={moe_config.moe_args.num_experts}, k={moe_config.moe_args.num_experts_per_token})")

dense_total, _ = count_parameters(model)
moe_total, _ = count_parameters(moe_model)
print(f"\ncrecimiento de parámetros: {moe_total / dense_total:.2f}x")

In [ ]:
history_moe = run_training(build_trainer(moe_model, "./checkpoints/tp1_moe"), epochs=EPOCHS)
plot_losses({"denso": history, "moe": history_moe}, title="denso vs MoE")
free()

# Consigna VII — utilización de expertos

Un MoE que rutea el 90% de sus tokens a un solo experto es un modelo denso que desperdicia
memoria. Medí si el tuyo realmente reparte la carga.

Usando `last_indices` (que guarda tu `MoELayer.forward`), corré unos pocos batches de
validación y, por capa, contá cuántos tokens fueron a cada experto. Graficalo como un
gráfico de barras, un grupo por capa, e imprimí la fracción de tokens que vio cada experto.

Una capa perfectamente balanceada manda `k / E` de los tokens a cada experto.

In [ ]:
# TODO: Consigna VII
@torch.no_grad()
def expert_utilization(model, loader, n_batches: int = 20):
    """
    Devuelve un tensor de forma (n_layer, num_experts) con la fracción de tokens ruteados
    a cada experto, y lo grafica.

    Pista: para cada bloque, model.blocks[i].ff.moe.last_indices tiene la selección (N, k)
    del forward más reciente. Usá torch.bincount.
    """
    raise NotImplementedError


utilization = expert_utilization(moe_model, val_loader)

## Preguntas

1. Si un experto queda sin tokens, explicá el bucle de realimentación que te llevó ahí.
   ¿Por qué se refuerza a sí mismo?
2. El ruteo es un `topk`, que no tiene gradiente útil. Entonces, ¿cómo aprende algo la
   gate? Trazá el camino desde la loss hasta `gate.proj.weight`.

---

-
-
-

# Consigna VIII — DeepSeekMoE

Tu MoE actual es un diseño legacy: `E` expertos iguales, elegir los `k` mejores. Leé
[DeepSeekMoE (Dai et al., 2024)](https://arxiv.org/abs/2401.06066) e implementá sus dos
cambios. Son los que usan DeepSeek-V2 y V3 y las versiones modernas, y los dos son deltas
chicos sobre la capa que ya escribiste.

**Segmentación fina de expertos.** Partí cada experto en `m` más chicos: multiplicá
`num_experts` y `num_experts_per_token` por `m`, y dividí el ancho oculto de cada experto
por `m`. Los parámetros totales y el cómputo por token no cambian. Lo que crece es la
cantidad de *combinaciones* de ruteo — de "elegir 2 de 4" a "elegir 4 de 8" — así que un
token puede expresar una mezcla más específica.

**Aislamiento de expertos compartidos.** Reservá `num_shared_experts` expertos por los que
todo token pasa, siempre, por fuera del top-k. El argumento: el conocimiento común a todos
los tokens (puntuación, espaciado, inglés básico) si no tiene que duplicarse dentro de cada
experto ruteado. Entregáselo a un experto compartido y los ruteados quedan libres para
especializarse.

Implementá `DeepSeekMoELayer`. Las dos configuraciones ya viven en `MoEArgs`
(`num_shared_experts`, `expert_hidden_mult`); `Expert` lee el ancho de la config, así que
no deberías necesitar tocarlo.

Después entrenalo contra tu MoE de la Consigna VI con **parámetros activos igualados** y
reportá loss, utilización de expertos, y tiempo por paso.

### Fijá tus expectativas primero

Escribí qué predecís, y recién después medí. Este modelo es de 2 capas con `n_embd=64`
entrenado sobre 100k caracteres. Segmentar 4 expertos en 8 deja a cada uno con un ancho
oculto de 128. Los resultados del paper son a 2B–145B parámetros sobre cientos de miles de
millones de tokens.

**Un resultado nulo acá es el resultado correcto, y explicarlo es la consigna.** Reproducir
una arquitectura de 2024 y después reportar honestamente que su beneficio no aparece a tu
escala vale más que un número que salió para el lado correcto por suerte.

In [ ]:
class DeepSeekMoELayer(nn.Module):
    """
    MoE con segmentación fina de expertos y expertos compartidos (arXiv:2401.06066).

    Los expertos compartidos corren para todo token, por fuera del top-k. Los ruteados se
    seleccionan como antes. Guardá last_indices / last_gate_probs sólo para los expertos
    ruteados, así el gráfico de utilización de la Consigna VII sigue funcionando.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        # TODO: Consigna VIII
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Consigna VIII
        raise NotImplementedError


class DeepSeekMoEFFN(nn.Module):
    """Reemplazo directo para config.ff_class, igual que MoEFFN."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.moe = DeepSeekMoELayer(config)

    def forward(self, x):
        return self.moe(x)


In [ ]:
# La segmentación fina es neutra en cómputo: 4 expertos x top-2 con oculto 4.0x pasa a
# 8 x top-4 con 2.0x, y los parámetros activos de FFN por token son idénticos en ambos
# casos. Lo que cambia es la cantidad de combinaciones de ruteo, de C(4,2)=6 a C(8,4)=70.
#
# El experto compartido NO es gratis -- corre para todo token, sumando el equivalente a un
# experto por encima. La pregunta 1 te pide calcular exactamente cuánto, y si eso invalida
# la comparación.
#
# SEGMENTS=2 mantiene esto accesible: 8 expertos ruteados + 1 compartido son 9 llamadas
# secuenciales a expertos por capa, contra 17 con SEGMENTS=4. El paper segmenta mucho más
# agresivamente -- poné 4 si tenés el cómputo y querés ver si cambia algo.
SEGMENTS = 2

ds_config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    ff_class=DeepSeekMoEFFN,
    moe_args=MoEArgs(
        num_experts=4 * SEGMENTS,
        num_experts_per_token=2 * SEGMENTS,
        num_shared_experts=1,
        expert_hidden_mult=4.0 / SEGMENTS,
    ),
)

ds_model = TinyGPT(ds_config).to(device)
if device == "cuda":
    ds_model = torch.compile(ds_model)

describe(moe_model, "MoE (Consigna VI)")
describe(ds_model, "DeepSeekMoE")


In [ ]:
history_ds = run_training(build_trainer(ds_model, "./checkpoints/tp1_deepseek"), epochs=EPOCHS)
plot_losses({"denso": history, "moe": history_moe, "deepseek": history_ds},
            title="denso vs MoE vs DeepSeekMoE")
free()


## Preguntas

1. Volvé a correr el gráfico de utilización de la Consigna VII sobre el modelo DeepSeekMoE.
   Con 8 expertos ruteados en vez de 4, ¿la carga queda más o menos balanceada, y por qué
   lo esperarías *antes* de mirar?
2. El experto compartido recibe todos los tokens. Inspeccioná la magnitud de su salida
   contra la de un experto ruteado. ¿Está haciendo lo que el paper afirma — absorber
   estructura común — o simplemente se convirtió en una segunda FFN densa?
3. El paper reporta ganancias claras; vos probablemente no las reprodujiste. Dá la razón
   más probable, y describí el experimento más chico que testearía si la escala es la
   explicación.

---

-
-
-
-
-


# Consigna IX — hacelo rápido

A libro abierto. No hay solución de referencia ni checker.

Lo mediste en la Consigna VI: el MoE entrena a aproximadamente **30x el costo por paso**
del modelo denso, para *menos* aritmética. La Consigna VIII lo empeoró — 16 expertos en vez
de 4 significa 16 llamadas secuenciales a expertos por capa.

**Hacelo más rápido. Reportá qué probaste, qué funcionó, qué no, y por qué.**

Reglas: la salida tiene que seguir siendo numéricamente equivalente a tu implementación de
la Consigna V (escribí un test que lo pruebe), y el modelo tiene que seguir entrenando. Más
allá de eso, vale todo.

Algunas direcciones, no una checklist — no estás obligado a usar ninguna:

- Cronometrá la *segunda* época, nunca la primera: la compilación perezosa de kernels hace
  que la época 1 sea unas 2.4x más lenta y va a favorecer o arruinar cualquier cambio que
  hagas.
- ¿Adónde se va realmente el tiempo? Perfilá antes de optimizar. `torch.profiler`, o
  cronometrá las partes a mano. La respuesta no está donde la mayoría supone.
- `torch.where` devuelve un tensor de largo variable, así que fuerza una sincronización
  GPU→CPU. La llamás una vez por experto por capa.
- Ordenar los tokens por experto una sola vez convierte un scatter en slices contiguos.
- Rellenar cada experto hasta una capacidad fija hace que las formas sean estáticas, que es
  lo que te permite batchear los expertos en un solo `torch.bmm`. ¿Qué te cuesta en
  correctitud el padding por capacidad?
- Los matmuls chicos dejan la GPU ociosa. ¿En cuántos lanzamientos de kernel, como mínimo,
  podría hacerse esto?

**Entregable:** tu implementación correcta más rápida, el test de equivalencia, una
medición antes/después en el mismo hardware, y un informe corto del razonamiento —
incluyendo las cosas que probaste que la hicieron más lenta. Esas suelen ser la parte más
informativa.



In [ ]:
# TODO: Consigna IX -- tu MoE correcto más rápido, más el test que prueba que sigue siendo correcto.


# Opcional — loss de balanceo de carga

El arreglo estándar para el colapso de ruteo es una loss auxiliar (Switch Transformer,
ec. 4):

$$L_{aux} = E \cdot \sum_{i=1}^{E} f_i \cdot P_i$$

donde $f_i$ es la fracción de tokens ruteados al experto $i$ y $P_i$ es la probabilidad
media que la gate le asigna al experto $i$. Se minimiza cuando ambas son uniformes.

Notá que $f_i$ viene de un `topk` y no lleva gradiente, mientras que $P_i$ viene de un
softmax y sí — el producto es lo que hace entrenable al término.

Implementala, agregá `total_loss = ce_loss + alpha * aux_loss` con `alpha` alrededor de
`0.01`, reentrená, y volvé a correr la Consigna IV. Vas a necesitar tu propio loop de
entrenamiento, o una `loss_fn` que meta mano en el modelo.

In [ ]:
# TODO: Opcional
def load_balancing_loss(model) -> torch.Tensor:
    """
    Loss auxiliar de Switch-Transformer, sumada sobre todas las capas MoE del modelo.

    Usa last_gate_probs (diferenciable) y last_indices (no) de cada capa.
    """
    raise NotImplementedError

## ¿Escribe mejor Shakespeare?

In [ ]:
for name, m in (("dense", model), ("moe", moe_model)):
    torch.manual_seed(0)
    print("=" * 70)
    print(f"### {name}")
    print(generate(m, tokenizer, "To be", max_new_tokens=250))

## Preguntas finales

1. El MoE tiene aproximadamente 2x los parámetros del modelo denso. ¿Cuánto de eso está
   *activo* por token? Calculalo: contá los parámetros de un experto, de una gate, y de
   todo lo que está fuera de la FFN, y sacá los parámetros activos para `k=2` y `E=4`.
2. ¿Cuándo un MoE es la elección equivocada? Dá dos escenarios concretos de despliegue.

---

-
-
-
-

---

# Consigna X — un tokenizer real

Entrenaste sobre 61 caracteres. Los modelos reales usan vocabularios subword de 30k–150k.
Esta consigna mide qué cambia eso, y produce el checkpoint que el TP-2 usa para fine-tuning
y que el notebook de serving convierte — así que no es opcional.

Leé esto antes de comparar nada:

> La loss de validación no es comparable entre tokenizers distintos. Es cross-entropy *por
> token*, y los dos modelos no se ponen de acuerdo en qué es un token. Un modelo subword
> reporta una loss *más alta* siendo el *mejor* modelo, porque cada uno de sus tokens
> carga alrededor de tres caracteres de información en vez de uno.
>
> La cantidad comparable es bits por carácter:
>
> $$\text{bpc} = \frac{\text{loss}}{\ln 2} \times \frac{\text{tokens}}{\text{caracteres}}$$
>
> Convertí primero, compará después.

Las celdas de abajo preentrenan la misma arquitectura sobre el mismo texto con el tokenizer
de GPT-2. Cambialo por otro si querés ver cuánto del resultado es específico de GPT-2.



## Por qué GPT-2, y no algo más chico

GPT-2 es la entrada canónica de BPE, no está gateado, y no requiere login. La alternativa
*usable* más chica es 49.152 (SmolLM, StarCoder), que no cambia nada de lo que importa acá.

Sólo se descargan archivos de tokenizer, unos pocos MB, no pesos del modelo.

In [ ]:
import math

from transformers import AutoTokenizer

sub_tokenizer = AutoTokenizer.from_pretrained("gpt2")

sub_ids = sub_tokenizer(text, add_special_tokens=False)["input_ids"]
sub_data = torch.tensor(sub_ids, dtype=torch.long)
sub_split = int(0.9 * len(sub_data))

# Caracteres detrás de cada split -- hacen falta para convertir loss en bits por carácter.
char_split = int(0.9 * len(text))
N_CHARS_VAL = len(text) - char_split

print(f"caracteres      {len(text):>9,}")
print(f"tokens char     {len(data):>9,}   vocab {tokenizer.vocab_size:>6,}")
print(f"tokens subword  {len(sub_data):>9,}   vocab {len(sub_tokenizer):>6,}")
print(f"compresión      {len(data) / len(sub_data):>9.2f}x")

sub_config = GPTConfig(vocab_size=len(sub_tokenizer))

sub_train_loader = DataLoader(CharDataset(sub_data[:sub_split], sub_config.block_size),
                              batch_size=sub_config.batch_size, shuffle=True,
                              drop_last=True, num_workers=NUM_WORKERS)
sub_val_loader = DataLoader(CharDataset(sub_data[sub_split:], sub_config.block_size),
                            batch_size=sub_config.batch_size, shuffle=False,
                            drop_last=True, num_workers=NUM_WORKERS)

print(f"\n{len(train_loader):>5,} pasos/época (char)")
print(f"{len(sub_train_loader):>5,} pasos/época (subword)")

In [ ]:
sub_model = TinyGPT(sub_config).to(device)
if device == "cuda":
    sub_model = torch.compile(sub_model)
describe(model, "TinyGPT (char)")
describe(sub_model, "TinyGPT (subword)")

sub_optimizer = make_optimizer(sub_model.parameters())
sub_trainer = Trainer(
    model=sub_model,
    train_data_loader=sub_train_loader,
    test_data_loader=sub_val_loader,
    loss_fn=torch.nn.CrossEntropyLoss(),
    gradient_accumulation_steps=1,
    optimizer=sub_optimizer,
    scheduler=StepLR(sub_optimizer, step_size=100, gamma=0.9),
    device=device,
    save_dir="./checkpoints/tp1_subword",
    save_every_n=500,
)

sub_history = run_training(sub_trainer, epochs=EPOCHS)
del sub_trainer
free()
plot_losses({"char": history, "subword": sub_history},
            title="misma arquitectura, mismo texto, distinto tokenizer")
# El TP-2 usa este checkpoint para fine-tuning de instrucciones
SUBWORD_CKPT = "./checkpoints/tp1_subword/checkpoint_final.pt"
print(f"\ncheckpoint subword -> {SUBWORD_CKPT}")
print("El TP-2 lo carga para fine-tuning de instrucciones; el notebook de serving lo convierte a GGUF.")


### NOTA

Guardá tu checkpoint en disco porque se reutiliza en el TP-II

## X.a — la métrica comparable

Implementá `bits_per_char`. Recibe una cross-entropy media en nats por token, la cantidad de
tokens del split sobre el que se midió, y la cantidad de caracteres que ese split cubre, y
devuelve bits por carácter.

Después armá la tabla de comparación: para cada modelo reportá loss de validación, bits por
token, y bits por carácter. Sólo la última columna es una comparación justa.

In [ ]:
# TODO: Consigna X
import math
def bits_per_char(loss_nats: float, n_tokens: int, n_chars: int) -> float:
    """
    Args:
        loss_nats: cross-entropy media por token, en nats (lo que devuelve CrossEntropyLoss).
        n_tokens:  cantidad de tokens del split sobre el que se midió la loss.
        n_chars:   cantidad de caracteres que cubre ese mismo split.
    """
    raise NotImplementedError


# El split de validación tiene len(data) - split tokens para el modelo de caracteres,
# y len(sub_data) - sub_split para el subword. Los dos cubren N_CHARS_VAL caracteres.
raise NotImplementedError("armar la tabla de comparación")

## X.b — ¿adónde se fueron los parámetros?

Los dos modelos tienen el mismo `n_embd`, `n_layer` y `n_head`, así que sus bloques
transformer son idénticos en tamaño. Todo lo demás que cambió vino del vocabulario.

Implementá `parameter_breakdown(model, vocab_size, train_ids)` reportando:

- parámetros totales,
- parámetros de embedding (`token_emb` + `head`) y su porcentaje del total,
- parámetros que no son de embedding (todo lo demás) — esto debería coincidir entre los dos
  modelos,
- cuántas filas distintas del vocabulario aparecen efectivamente en los datos de
  entrenamiento, y qué fracción del vocabulario es eso.

Corrélo para los dos modelos. El último número es el que hay que pensar más en serio.

In [ ]:
# TODO: Consigna X
def parameter_breakdown(model, vocab_size: int, train_ids, name: str = ""):
    """Separa los parámetros del modelo en embedding vs no-embedding, y reporta la cobertura del vocabulario."""
    raise NotImplementedError


parameter_breakdown(model, tokenizer.vocab_size, data[:split].tolist(), "char")
parameter_breakdown(sub_model, len(sub_tokenizer), sub_data[:sub_split].tolist(), "subword")

In [ ]:
for name, m, tok in (("char", model, tokenizer), ("subword", sub_model, sub_tokenizer)):
    torch.manual_seed(0)
    print("=" * 70)
    print(f"### {name}")
    print(generateV2(m, tok, "To be", max_new_tokens=120, do_sample=True,
                     temperature=0.8, top_p=0.9))

## Preguntas

1. Ordená los dos modelos por loss de validación, y después por bits por carácter. ¿Los dos
   ordenamientos coinciden? Explicá en una oración por qué la loss cruda es el eje
   equivocado, usando la relación tokens/carácter que mediste.
2. Le diste al modelo subword ~31x los parámetros y vio el mismo texto. ¿Es una comparación
   justa? Nombrá el confundidor y proponé un experimento específico que lo elimine.
3. El modelo subword corre ~430 pasos por época contra los ~1.400 del modelo de caracteres,
   sobre el mismo texto. ¿De dónde viene el ahorro, y qué costó por paso?
4. 100.000 caracteres son 61k tokens subword, repartidos en ~5.500 tipos distintos —
   aproximadamente 11 ocurrencias cada uno. El modelo de caracteres tiene 200k tokens sobre
   62 tipos. ¿Qué tokenizer esperarías que gane a esta escala de datos, y cuál a 1000x más
   datos? ¿Por qué?
5. Leé las dos muestras. Las dos son malas, pero son malas de maneras *distintas*. Describí
   la diferencia y conectala con lo que cada tokenizer hace fácil o difícil de representar.

---

-
-
-
-
-
-

# Conclusiones

-
-
-


# ¡Felicitaciones! 🎉

Preentrenaste un GPT desde cero, implementaste las estrategias de decodificación que trae
todo LLM en producción, e implementaste cómputo condicional sparse — la única idea
arquitectónica que permite que los modelos frontier crezcan en cantidad de parámetros más
rápido que en costo de inferencia.

Lo que sigue: el TP-2, donde tomás este modelo y le enseñás a seguir instrucciones.

¡Ahora andá a alardear con tus amigos sobre cómo funcionan los LLMs y los GPTs!

